<a href="https://colab.research.google.com/github/2303A51528/reinforcement_learning_1528/blob/main/practical_10_rl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 10: Bellman Optimality and Value Iteration in a Deterministic Grid World

This notebook implements the Bellman Optimality equation using Value Iteration for a 2x2 Deterministic Grid World containing two non-terminal states ($S_0$, $S_1$), a wall, and a terminal goal state $G$.

In [4]:
import pandas as pd
import numpy as np

# --- TASK 1: Inspect the MDP configuration ---
gamma = 0.9
theta = 1e-6

# Define States
# S0: (0,0), S1: (0,1), Wall: (1,0) [Impassable], G: (1,1) [Terminal]
states = ['S0', 'S1', 'G']
non_terminal_states = ['S0', 'S1']
actions = ['UP', 'DOWN', 'LEFT', 'RIGHT']

# Transition and reward function: P(s, a) -> (next_state, reward)
# S0 is at (0,0), S1 is at (0,1). Wall is at (1,0). G is at (1,1).
transitions = {
    'S0': {
        'UP': ('S0', -1),       # Hits outer boundary
        'DOWN': ('S0', -1),     # Hits the Wall at (1,0)
        'LEFT': ('S0', -1),     # Hits outer boundary
        'RIGHT': ('S1', -1)     # Moves to S1
    },
    'S1': {
        'UP': ('S1', -1),       # Hits outer boundary
        'DOWN': ('G', 10),      # Reaches the Goal
        'LEFT': ('S0', -1),     # Moves to S0
        'RIGHT': ('S1', -1)     # Hits outer boundary
    }
}

print("=== Task 1: MDP Configuration ===")
print(f"States: {states}")
print(f"Non-Terminal States: {non_terminal_states}")
print(f"Actions: {actions}")
print(f"Discount Factor (gamma): {gamma}")
print(f"Convergence Threshold (theta): {theta}")

=== Task 1: MDP Configuration ===
States: ['S0', 'S1', 'G']
Non-Terminal States: ['S0', 'S1']
Actions: ['UP', 'DOWN', 'LEFT', 'RIGHT']
Discount Factor (gamma): 0.9
Convergence Threshold (theta): 1e-06


In [5]:
# --- TASK 2: Verify Transitions ---
print("=== Task 2: Transition & Reward Verification ===")
transition_records = []
for s in non_terminal_states:
    for a in actions:
        next_s, reward = transitions[s][a]
        transition_records.append({'Current State': s, 'Action': a, 'Next State': next_s, 'Reward': reward})

df_transitions = pd.DataFrame(transition_records)
display(df_transitions)

=== Task 2: Transition & Reward Verification ===


,Current State,Action,Next State,Reward
0,S0,UP,S0,-1
1,S0,DOWN,S0,-1
2,S0,LEFT,S0,-1
3,S0,RIGHT,S1,-1
4,S1,UP,S1,-1
5,S1,DOWN,G,10
6,S1,LEFT,S0,-1
7,S1,RIGHT,S1,-1


In [6]:
# --- TASK 3: Initialize State Values ---
V = {s: 0.0 for s in states}
print("=== Task 3: Initialized State Values ===")
for s, val in V.items():
    print(f"V({s}) = {val}")

=== Task 3: Initialized State Values ===
V(S0) = 0.0
V(S1) = 0.0
V(G) = 0.0


In [7]:
# --- TASK 4 & 5: Implement Value Iteration (Synchronous) ---
print("=== Task 4 & 5: Value Iteration Progress ===")
iteration = 0
converged = False

# Track history for reporting
history = [{'Iteration': 0, 'V(S0)': V['S0'], 'V(S1)': V['S1'], 'V(G)': V['G'], 'Delta': '-'}]

while not converged:
    iteration += 1
    V_new = V.copy() # Synchronous update: hold new values in a temporary dict
    max_delta = 0

    for s in non_terminal_states:
        action_values = []
        for a in actions:
            next_s, reward = transitions[s][a]
            q_val = reward + gamma * V[next_s]
            action_values.append(q_val)

        # Bellman Optimality Update
        V_new[s] = max(action_values)
        max_delta = max(max_delta, abs(V_new[s] - V[s]))

    # Target G state remains 0 as it is terminal
    V_new['G'] = 0.0

    V = V_new
    history.append({
        'Iteration': iteration,
        'V(S0)': round(V['S0'], 5),
        'V(S1)': round(V['S1'], 5),
        'V(G)': round(V['G'], 5),
        'Delta': round(max_delta, 5)
    })

    if max_delta < theta:
        converged = True

df_history = pd.DataFrame(history)
display(df_history)

=== Task 4 & 5: Value Iteration Progress ===


,Iteration,V(S0),V(S1),V(G),Delta
0,0,0.0,0.0,0.0,-
1,1,-1.0,10.0,0.0,10.0
2,2,8.0,10.0,0.0,9.0
3,3,8.0,10.0,0.0,0


In [8]:
# --- TASK 6: Calculate Action-Values Q*(s, a) ---
print("=== Task 6: Final Action Values Q*(s, a) ===")
q_records = []
for s in non_terminal_states:
    for a in actions:
        next_s, reward = transitions[s][a]
        q_val = reward + gamma * V[next_s]
        q_records.append({'State': s, 'Action': a, 'Q*(s,a)': round(q_val, 5)})

df_q = pd.DataFrame(q_records)
display(df_q)

=== Task 6: Final Action Values Q*(s, a) ===


,State,Action,"Q*(s,a)"
0,S0,UP,6.2
1,S0,DOWN,6.2
2,S0,LEFT,6.2
3,S0,RIGHT,8.0
4,S1,UP,8.0
5,S1,DOWN,10.0
6,S1,LEFT,6.2
7,S1,RIGHT,8.0


In [9]:
# --- TASK 7: Extract Optimal Policy ---
print("=== Task 7: Optimal Policy extraction ===")
optimal_policy = {}
for s in non_terminal_states:
    s_q = df_q[df_q['State'] == s]
    max_q = s_q['Q*(s,a)'].max()
    best_actions = s_q[np.isclose(s_q['Q*(s,a)'], max_q)]['Action'].tolist()
    optimal_policy[s] = (best_actions, round(max_q, 5))
    print(f"State {s}: Best Action(s) = {best_actions} with Q* = {round(max_q, 5)}")

=== Task 7: Optimal Policy extraction ===
State S0: Best Action(s) = ['RIGHT'] with Q* = 8.0
State S1: Best Action(s) = ['DOWN'] with Q* = 10.0


In [10]:
# --- TASK 8: Verify Bellman Optimality ---
print("=== Task 8: Verification of Bellman Optimality [ V*(s) == max_a Q*(s,a) ] ===")
verification_records = []
for s in non_terminal_states:
    v_star = V[s]
    max_q_star = optimal_policy[s][1]
    is_verified = np.isclose(v_star, max_q_star)
    verification_records.append({
        'State': s,
        'V*(s)': round(v_star, 5),
        'max_a Q*(s,a)': max_q_star,
        'Verified': is_verified
    })

df_verify = pd.DataFrame(verification_records)
display(df_verify)

=== Task 8: Verification of Bellman Optimality [ V*(s) == max_a Q*(s,a) ] ===


,State,V*(s),"max_a Q*(s,a)",Verified
0,S0,8.0,8.0,True
1,S1,10.0,10.0,True


### Task 9: Explanation of the Results

1. **Optimal Actions & Policy**:
   - **For State $S_1$**: The optimal action is `DOWN` (value: $+10.0$) because it leads immediately to the terminal goal state $G$, returning the $+10$ reward without discounting.
   - **For State $S_0$**: The optimal action is `RIGHT` because transitioning to $S_1$ puts the agent one step closer to the reward. $V^*(S_0)$ evaluates to $8.0$.

2. **Influence of Goal Reward**:
   - The goal reward at $G$ increases the value of neighboring states. This effect propagates step-by-step backward across the deterministic transition paths.

3. **Role of Discount Factor ($\gamma = 0.9$)**:
   - It reduces the valuation of future rewards, ensuring the algorithm converges mathematically and encouraging the selection of shorter paths.